# 07 - Inferencia del modelo Dish NER y generación de menciones de platos

## Objetivo

Este notebook aplica el modelo `Dish NER v1` entrenado en el Notebook 06 sobre el corpus de reviews gastronómicas.

El objetivo es transformar reseñas completas en una tabla estructurada de menciones de platos:

```text
review_text
→ modelo NER
→ entidades DISH detectadas
→ dish_mentions_model_v1.jsonl

```

In [1]:
# ============================================================
# 01. Instalación de dependencias
# ============================================================

!pip -q install transformers datasets accelerate pandas numpy tqdm pyarrow

In [2]:
# ============================================================
# 02. Imports y configuración base
# ============================================================

from pathlib import Path
import json
import random
import re
import os
import shutil
import zipfile
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 300)

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA disponible: True
GPU: Tesla T4


In [3]:
# ============================================================
# 03. Detectar entorno y preparar carpetas
# ============================================================

IS_KAGGLE = Path("/kaggle/input").exists()
IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False

if IS_KAGGLE:
    ENV_NAME = "kaggle"
elif IS_COLAB:
    ENV_NAME = "colab"
else:
    ENV_NAME = "local"

print("Entorno detectado:", ENV_NAME)

if IS_KAGGLE:
    INPUT_DIR = Path("/kaggle/input")
    WORKING_DIR = Path("/kaggle/working")
    PROJECT_DIR = WORKING_DIR / "hidden_gems_ai"

elif IS_COLAB:
    from google.colab import drive

    try:
        !fusermount -u /content/drive 2>/dev/null || true
        !rm -rf /content/drive

        drive.mount("/content/drive", force_remount=True, timeout_ms=120000)

        PROJECT_DIR = Path("/content/drive/MyDrive/hidden_gems_ai")

    except Exception as e:
        print("No se ha podido montar Drive. Se usará almacenamiento temporal.")
        print("Error:", e)

        PROJECT_DIR = Path("/content/hidden_gems_ai")

else:
    PROJECT_DIR = Path.cwd()

OUTPUT_DIR = PROJECT_DIR / "outputs" / "dish_ner_inference"
MENTIONS_DIR = OUTPUT_DIR / "mentions"
PREDICTIONS_DIR = OUTPUT_DIR / "predictions"
ARTIFACTS_DIR = OUTPUT_DIR / "artifacts"
SAMPLES_DIR = OUTPUT_DIR / "samples"
TMP_DIR = PROJECT_DIR / "tmp"

for path in [PROJECT_DIR, OUTPUT_DIR, MENTIONS_DIR, PREDICTIONS_DIR, ARTIFACTS_DIR, SAMPLES_DIR, TMP_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

Entorno detectado: kaggle
PROJECT_DIR: /kaggle/working/hidden_gems_ai
OUTPUT_DIR: /kaggle/working/hidden_gems_ai/outputs/dish_ner_inference


In [4]:
# ============================================================
# 04. Diagnóstico de archivos disponibles
# ============================================================

if IS_KAGGLE:
    print("Archivos disponibles en /kaggle/input:")

    for dirname, _, filenames in os.walk("/kaggle/input"):
        for filename in filenames:
            print(os.path.join(dirname, filename))
else:
    print("No estás en Kaggle. Se buscarán archivos en el proyecto local.")

Archivos disponibles en /kaggle/input:
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_labels.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_bio_high_precision_with_negatives.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_detection_manual_review_sample_v2_150.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/transformer_sentiment_metrics.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/weak_dish_candidates_v2.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_detection_exploration_summary_v2.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/yelp_translation_gold_seed_300_es_ready_v1.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_bio_smoke_test.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/weak_dish_candidate_frequencies_v2.csv
/kaggle/input/datasets/ivanarteagacordero/hidden-

In [5]:
# ============================================================
# 05. Localizar corpus y modelo entrenado
# ============================================================

CORPUS_FILENAME = "yelp_food_reviews_corpus_sample_100k_lines.jsonl"
MODEL_FOLDER_NAME = "dish_ner_transformer_full"
MODEL_ZIP_NAME = "dish_ner_transformer_full.zip"

def find_file(filename: str) -> Path:
    candidate_paths = []

    if IS_KAGGLE:
        candidate_paths.extend(list(Path("/kaggle/input").rglob(filename)))

    candidate_paths.extend(list(PROJECT_DIR.rglob(filename)))
    candidate_paths.extend(list(Path.cwd().rglob(filename)))

    candidate_paths = list(dict.fromkeys(candidate_paths))

    if not candidate_paths:
        raise FileNotFoundError(
            f"No se ha encontrado el archivo: {filename}\n"
            "En Kaggle, asegúrate de haberlo añadido desde Add Data."
        )

    return candidate_paths[0]


def find_model_dir(folder_name: str) -> Path | None:
    candidate_dirs = []

    if IS_KAGGLE:
        candidate_dirs.extend([
            path for path in Path("/kaggle/input").rglob(folder_name)
            if path.is_dir()
        ])

    candidate_dirs.extend([
        path for path in PROJECT_DIR.rglob(folder_name)
        if path.is_dir()
    ])

    candidate_dirs.extend([
        path for path in Path.cwd().rglob(folder_name)
        if path.is_dir()
    ])

    # Validar que parece un modelo Hugging Face
    valid_dirs = []

    for path in candidate_dirs:
        has_config = (path / "config.json").exists()
        has_model = (path / "model.safetensors").exists() or (path / "pytorch_model.bin").exists()

        if has_config and has_model:
            valid_dirs.append(path)

    if valid_dirs:
        return valid_dirs[0]

    return None


CORPUS_PATH = find_file(CORPUS_FILENAME)

MODEL_PATH = find_model_dir(MODEL_FOLDER_NAME)

if MODEL_PATH is None:
    print("No se encontró carpeta de modelo. Buscando ZIP...")

    try:
        model_zip_path = find_file(MODEL_ZIP_NAME)

        unzip_dir = TMP_DIR / MODEL_FOLDER_NAME

        if unzip_dir.exists():
            shutil.rmtree(unzip_dir)

        unzip_dir.mkdir(parents=True, exist_ok=True)

        print("Descomprimiendo modelo desde:")
        print(model_zip_path)

        with zipfile.ZipFile(model_zip_path, "r") as zip_ref:
            zip_ref.extractall(unzip_dir)

        # Caso A: el ZIP contiene directamente los archivos del modelo.
        if (unzip_dir / "config.json").exists():
            MODEL_PATH = unzip_dir

        # Caso B: el ZIP contiene una carpeta interna.
        else:
            nested_model_dirs = [
                path for path in unzip_dir.rglob("*")
                if path.is_dir()
                and (path / "config.json").exists()
                and ((path / "model.safetensors").exists() or (path / "pytorch_model.bin").exists())
            ]

            if not nested_model_dirs:
                raise FileNotFoundError(
                    "El ZIP se descomprimió, pero no se encontró una carpeta válida de modelo Hugging Face."
                )

            MODEL_PATH = nested_model_dirs[0]

    except FileNotFoundError:
        raise FileNotFoundError(
            "No se encontró ni la carpeta del modelo ni el ZIP.\n"
            "Debes añadir a Kaggle el modelo `dish_ner_transformer_full/` "
            "o el archivo `dish_ner_transformer_full.zip`."
        )

print("CORPUS_PATH:", CORPUS_PATH)
print("MODEL_PATH:", MODEL_PATH)

print("\nArchivos del modelo:")
for item in MODEL_PATH.iterdir():
    print("-", item.name)

CORPUS_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/yelp_food_reviews_corpus_sample_100k_lines.jsonl
MODEL_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full

Archivos del modelo:
- config.json
- training_args.bin
- tokenizer.json
- tokenizer_config.json
- model.safetensors


In [6]:
# ============================================================
# 06. Configuración de inferencia
# ============================================================

MAX_LENGTH = 256
BATCH_SIZE = 32

# Para probar primero el flujo.
# Cuando funcione, cambia a False para inferir sobre todo el corpus.
FAST_DEV_RUN = False

FAST_DEV_N = 1000

RUN_NAME = "fast_dev" if FAST_DEV_RUN else "full"

print("MAX_LENGTH:", MAX_LENGTH)
print("BATCH_SIZE:", BATCH_SIZE)
print("FAST_DEV_RUN:", FAST_DEV_RUN)
print("RUN_NAME:", RUN_NAME)

MAX_LENGTH: 256
BATCH_SIZE: 32
FAST_DEV_RUN: False
RUN_NAME: full


In [7]:
# ============================================================
# 07. Cargar corpus JSONL
# ============================================================

def load_jsonl(path: Path, max_records: int | None = None) -> pd.DataFrame:
    records = []
    invalid_json_count = 0

    with path.open("r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Leyendo {path.name}"):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                invalid_json_count += 1
                continue

            if max_records is not None and len(records) >= max_records:
                break

    print(f"Archivo: {path.name}")
    print(f"Registros cargados: {len(records):,}")
    print(f"Líneas JSON inválidas: {invalid_json_count:,}")

    return pd.DataFrame(records)


max_records = FAST_DEV_N if FAST_DEV_RUN else None

df_raw = load_jsonl(CORPUS_PATH, max_records=max_records)

print("Shape raw:", df_raw.shape)
display(df_raw.head(3))

Leyendo yelp_food_reviews_corpus_sample_100k_lines.jsonl: 0it [00:00, ?it/s]

Archivo: yelp_food_reviews_corpus_sample_100k_lines.jsonl
Registros cargados: 79,270
Líneas JSON inválidas: 0
Shape raw: (79270, 20)


,corpus_document_id,source_system_code,source_dataset,source_entity_type,source_review_id,source_business_id,source_user_id,text,text_normalized,language,rating_value,sentiment_label_from_rating,review_date,corpus_split,task_scope,is_training_eligible,quality_flags,business_metadata,source_metrics,created_at
0,yelp_2fbfd094613536a7b8c9231b,yelp_open_dataset,yelp_open_dataset,yelp_review,KU_O5udG6zpxOg-VcAEodg,XQfwVwDr-v0ZS3_CbbE5Xw,mh_-eMZ6K5RLWhZyISBhwA,"If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is good, but it takes a very long time to come out. The...","If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is good, but it takes a very long time to come out. The...",en,3.0,neutral,2018-07-07 22:09:11,train,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 511, 'label_is_weak': True, 'label_source': 'rating_value'}","{'business_name': 'Turning Point of North Wales', 'city': 'North Wales', 'state': 'PA', 'stars_business': 3.0, 'review_count_business': 169, 'is_open': 1, 'categories_list': ['Restaurants', 'Breakfast & Brunch', 'Food', 'Juice Bars & Smoothies', 'American (New)', 'Coffee & Tea', 'Sandwiches'], '...","{'useful_count': 0, 'funny_count': 0, 'cool_count': 0}",2026-05-06T10:57:56.623430+00:00
1,yelp_94c5a64cecd4448d105e5c8a,yelp_open_dataset,yelp_open_dataset,yelp_review,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,8g_iMtfSiwikVnbP2etR0A,"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ...","Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ...",en,3.0,neutral,2014-02-05 20:30:30,train,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 339, 'label_is_weak': True, 'label_source': 'rating_value'}","{'business_name': 'Kettle Restaurant', 'city': 'Tucson', 'state': 'AZ', 'stars_business': 3.5, 'review_count_business': 47, 'is_open': 1, 'categories_list': ['Restaurants', 'Breakfast & Brunch'], 'food_category_tags': ['Breakfast & Brunch', 'Restaurants'], 'food_confidence': 0.95}","{'useful_count': 0, 'funny_count': 0, 'cool_count': 0}",2026-05-06T10:57:56.624430+00:00
2,yelp_69e10d25d69774ab39af6571,yelp_open_dataset,yelp_open_dataset,yelp_review,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,_7bHUi9Uuf5__HHc_Q8guQ,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!","Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",en,5.0,positive,2015-01-04 00:01:03,train,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 235, 'label_is_weak': Tru

In [8]:
# ============================================================
# 08. Preparar dataframe de inferencia
# ============================================================

required_cols = [
    "corpus_document_id",
    "source_review_id",
    "source_business_id",
    "text",
    "text_normalized",
    "language",
    "rating_value",
    "sentiment_label_from_rating",
    "corpus_split",
]

missing_cols = [col for col in required_cols if col not in df_raw.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas obligatorias en el corpus: {missing_cols}")

df = df_raw.copy()

df["corpus_document_id"] = df["corpus_document_id"].astype(str)
df["review_id"] = df["source_review_id"].astype(str)
df["business_id"] = df["source_business_id"].astype(str)

df["text"] = (
    df["text_normalized"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["text_original"] = (
    df["text"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["language"] = df["language"].astype(str).str.lower().str.strip()
df["rating_value"] = pd.to_numeric(df["rating_value"], errors="coerce")

df["sentiment_label"] = (
    df["sentiment_label_from_rating"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["split"] = (
    df["corpus_split"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["text_length_chars"] = df["text"].str.len()

df = df[
    df["text"].notna()
    & (df["text"].str.len() > 0)
].copy()

print("Shape preparado:", df.shape)

print("\nSplit counts:")
display(df["split"].value_counts())

print("\nSentiment counts:")
display(df["sentiment_label"].value_counts())

display(df.head(3))

Shape preparado: (79270, 26)

Split counts:


split
train         63540
test           7881
validation     7849
Name: count, dtype: int64


Sentiment counts:


sentiment_label
positive    54857
negative    14578
neutral      9835
Name: count, dtype: int64

,corpus_document_id,source_system_code,source_dataset,source_entity_type,source_review_id,source_business_id,source_user_id,text,text_normalized,language,rating_value,sentiment_label_from_rating,review_date,corpus_split,task_scope,is_training_eligible,quality_flags,business_metadata,source_metrics,created_at,review_id,business_id,text_original,sentiment_label,split,text_length_chars
0,yelp_2fbfd094613536a7b8c9231b,yelp_open_dataset,yelp_open_dataset,yelp_review,KU_O5udG6zpxOg-VcAEodg,XQfwVwDr-v0ZS3_CbbE5Xw,mh_-eMZ6K5RLWhZyISBhwA,"If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is good, but it takes a very long time to come out. The...","If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is good, but it takes a very long time to come out. The...",en,3.0,neutral,2018-07-07 22:09:11,train,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 511, 'label_is_weak': True, 'label_source': 'rating_value'}","{'business_name': 'Turning Point of North Wales', 'city': 'North Wales', 'state': 'PA', 'stars_business': 3.0, 'review_count_business': 169, 'is_open': 1, 'categories_list': ['Restaurants', 'Breakfast & Brunch', 'Food', 'Juice Bars & Smoothies', 'American (New)', 'Coffee & Tea', 'Sandwiches'], '...","{'useful_count': 0, 'funny_count': 0, 'cool_count': 0}",2026-05-06T10:57:56.623430+00:00,KU_O5udG6zpxOg-VcAEodg,XQfwVwDr-v0ZS3_CbbE5Xw,"If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is good, but it takes a very long time to come out. The...",neutral,train,511
1,yelp_94c5a64cecd4448d105e5c8a,yelp_open_dataset,yelp_open_dataset,yelp_review,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,8g_iMtfSiwikVnbP2etR0A,"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ...","Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ...",en,3.0,neutral,2014-02-05 20:30:30,train,"[dish_extraction, review_sentiment, dish_sentiment, recommendation_signal]",True,"{'has_text': True, 'has_rating': True, 'has_business_metadata': True, 'text_length_chars': 339, 'label_is_weak': True, 'label_source': 'rating_value'}","{'business_name': 'Kettle Restaurant', 'city': 'Tucson', 'state': 'AZ', 'stars_business': 3.5, 'review_count_business': 47, 'is_open': 1, 'categories_list': ['Restaurants', 'Breakfast & Brunch'], 'food_category_tags': ['Breakfast & Brunch', 'Restaurants'], 'food_confidence': 0.95}","{'useful_count': 0, 'funny_count': 0, 'cool_count': 0}",2026-05-06T10:57:56.624430+00:00,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ...",neutral,train,339
2,yelp_69e10d25d69774ab39af6571,yelp_open_dataset,yelp_ope

In [9]:
# ============================================================
# 09. Extraer metadata de negocio
# ============================================================

def parse_maybe_json(value):
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            return {}

    return {}


def safe_list(value):
    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

        return [part.strip() for part in value.split(",") if part.strip()]

    return []


business_names = []
cities = []
states = []
food_category_tags = []
business_stars = []
business_review_counts = []

for value in df.get("business_metadata", [{}]):
    metadata = parse_maybe_json(value)

    business_names.append(
        metadata.get("business_name")
        or metadata.get("name")
        or metadata.get("source_name_raw")
        or ""
    )

    cities.append(metadata.get("city") or "")
    states.append(metadata.get("state") or "")

    tags = (
        metadata.get("food_category_tags")
        or metadata.get("categories_list")
        or metadata.get("categories")
        or []
    )

    food_category_tags.append(safe_list(tags))

    business_stars.append(metadata.get("stars") or metadata.get("business_stars") or None)
    business_review_counts.append(metadata.get("review_count") or metadata.get("business_review_count") or None)

df["business_name"] = business_names
df["city"] = cities
df["state"] = states
df["food_category_tags"] = food_category_tags
df["business_stars"] = business_stars
df["business_review_count"] = business_review_counts

display(
    df[
        [
            "review_id",
            "business_id",
            "business_name",
            "city",
            "state",
            "rating_value",
            "sentiment_label",
            "split",
            "text"
        ]
    ].head(5)
)

,review_id,business_id,business_name,city,state,rating_value,sentiment_label,split,text
0,KU_O5udG6zpxOg-VcAEodg,XQfwVwDr-v0ZS3_CbbE5Xw,Turning Point of North Wales,North Wales,PA,3.0,neutral,train,"If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is good, but it takes a very long time to come out. The..."
1,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,Kettle Restaurant,Tucson,AZ,3.0,neutral,train,"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ..."
2,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,5.0,positive,train,"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!"
3,Sx8TMOWLNuJBWer-0pcmoA,e4Vwtrqf-wpJfwesgvdgxQ,Melt,New Orleans,LA,4.0,positive,train,"Cute interior and owner (?) gave us tour of upcoming patio/rooftop area which will be great on beautiful days like today. Cheese curds were very good and very filling. Really like that sandwiches come w salad, esp after eating too many curds! Had the onion, gruyere, tomato sandwich. Wasn't too m..."
4,JrIxlS1TzJ-iCu79ul40cQ,04UD14gamNjLY0IDYVhHJg,Dmitri's,Philadelphia,PA,1.0,negative,train,I am a long term frequent customer of this establishment. I just went in to order take out (3 apps) and was told they're too busy to do it. Really? The place is maybe half full at best. Does your dick reach your ass? Yes? Go fuck yourself! I'm a frequent customer AND great tipper. Glad that Kane...


In [10]:
# ============================================================
# 10. Cargar modelo y tokenizer
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH))
model = AutoModelForTokenClassification.from_pretrained(str(MODEL_PATH))

model.to(device)
model.eval()

id2label = {
    int(k): v
    for k, v in model.config.id2label.items()
}

label2id = {
    v: int(k)
    for k, v in id2label.items()
}

print("Modelo cargado correctamente.")
print("Device:", device)
print("Labels:", id2label)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Modelo cargado correctamente.
Device: cuda
Labels: {0: 'O', 1: 'B-DISH', 2: 'I-DISH'}


In [11]:
# ============================================================
# 11. Tokenizador simple para palabras
# ============================================================

SIMPLE_TOKEN_PATTERN = re.compile(
    r"\w+(?:[-']\w+)*|[^\w\s]",
    flags=re.UNICODE
)

def simple_tokenize_with_offsets(text: str) -> list[dict]:
    """
    Tokeniza texto en palabras/puntuación y conserva offsets de caracteres.
    """

    if not isinstance(text, str):
        return []

    tokens = []

    for match in SIMPLE_TOKEN_PATTERN.finditer(text):
        tokens.append({
            "text": match.group(0),
            "start": int(match.start()),
            "end": int(match.end()),
        })

    return tokens


def extract_entities_from_bio_with_offsets(tokens_with_offsets, labels, token_confidences=None):
    """
    Extrae entidades DISH desde tokens + etiquetas BIO + offsets.
    """

    entities = []
    current = None

    if token_confidences is None:
        token_confidences = [None] * len(labels)

    for idx, (token_info, label, confidence) in enumerate(zip(tokens_with_offsets, labels, token_confidences)):
        token_text = token_info["text"]

        if label == "B-DISH":
            if current is not None:
                entities.append(current)

            current = {
                "label": "DISH",
                "start_token": idx,
                "end_token": idx + 1,
                "start_char": int(token_info["start"]),
                "end_char": int(token_info["end"]),
                "tokens": [token_text],
                "token_confidences": [] if confidence is None else [float(confidence)],
            }

        elif label == "I-DISH":
            if current is not None:
                current["end_token"] = idx + 1
                current["end_char"] = int(token_info["end"])
                current["tokens"].append(token_text)

                if confidence is not None:
                    current["token_confidences"].append(float(confidence))

            else:
                # Caso defensivo: I-DISH sin B-DISH previa.
                current = {
                    "label": "DISH",
                    "start_token": idx,
                    "end_token": idx + 1,
                    "start_char": int(token_info["start"]),
                    "end_char": int(token_info["end"]),
                    "tokens": [token_text],
                    "token_confidences": [] if confidence is None else [float(confidence)],
                }

        else:
            if current is not None:
                entities.append(current)
                current = None

    if current is not None:
        entities.append(current)

    for ent in entities:
        ent["text"] = " ".join(ent["tokens"])
        ent["normalized_text"] = ent["text"].lower()

        if ent["token_confidences"]:
            ent["confidence_mean"] = float(np.mean(ent["token_confidences"]))
            ent["confidence_min"] = float(np.min(ent["token_confidences"]))
            ent["confidence_max"] = float(np.max(ent["token_confidences"]))
        else:
            ent["confidence_mean"] = None
            ent["confidence_min"] = None
            ent["confidence_max"] = None

    return entities


def render_tokens_with_entities(tokens, labels):
    output = []
    inside = False

    for token, label in zip(tokens, labels):
        if label == "B-DISH":
            if inside:
                output.append("]")
            output.append("[DISH:")
            output.append(token)
            inside = True

        elif label == "I-DISH":
            if inside:
                output.append(" " + token)
            else:
                output.append("[DISH:" + token)
                inside = True

        else:
            if inside:
                output.append("]")
                inside = False

            output.append(" " + token)

    if inside:
        output.append("]")

    return "".join(output).strip()

In [12]:
# ============================================================
# 12. Función de inferencia por lotes
# ============================================================

def predict_batch_texts(texts: list[str]):
    """
    Predice entidades DISH para una lista de textos.
    Devuelve una lista de resultados por texto.
    """

    batch_tokens_with_offsets = [
        simple_tokenize_with_offsets(text)
        for text in texts
    ]

    batch_tokens = [
        [token["text"] for token in tokens_with_offsets]
        for tokens_with_offsets in batch_tokens_with_offsets
    ]

    # Importante:
    # Mantenemos este objeto como BatchEncoding para poder usar word_ids().
    encoding = tokenizer(
        batch_tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors="pt"
    )

    # Creamos una copia en device para el modelo,
    # pero conservamos encoding original para word_ids().
    model_inputs = {
        key: value.to(device)
        for key, value in encoding.items()
    }

    with torch.no_grad():
        outputs = model(**model_inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=-1)
        pred_ids = torch.argmax(probabilities, dim=-1)

    pred_ids_np = pred_ids.cpu().numpy()
    probabilities_np = probabilities.cpu().numpy()

    results = []

    for batch_index, tokens_with_offsets in enumerate(batch_tokens_with_offsets):
        word_ids = encoding.word_ids(batch_index=batch_index)

        word_predictions = {}
        word_confidences = {}

        previous_word_id = None

        for subtoken_index, word_id in enumerate(word_ids):
            if word_id is None:
                continue

            # Solo usamos el primer subtoken de cada palabra,
            # coherente con el entrenamiento LABEL_ALL_TOKENS=False.
            if word_id != previous_word_id:
                pred_id = int(pred_ids_np[batch_index][subtoken_index])
                label = id2label[pred_id]
                confidence = float(probabilities_np[batch_index][subtoken_index][pred_id])

                word_predictions[word_id] = label
                word_confidences[word_id] = confidence

            previous_word_id = word_id

        supervised_token_count = max(word_predictions.keys()) + 1 if word_predictions else 0

        visible_tokens_with_offsets = tokens_with_offsets[:supervised_token_count]
        visible_tokens = [token["text"] for token in visible_tokens_with_offsets]

        pred_labels = [
            word_predictions.get(i, "O")
            for i in range(supervised_token_count)
        ]

        pred_confidences = [
            word_confidences.get(i, None)
            for i in range(supervised_token_count)
        ]

        entities = extract_entities_from_bio_with_offsets(
            visible_tokens_with_offsets,
            pred_labels,
            pred_confidences
        )

        rendered = render_tokens_with_entities(visible_tokens, pred_labels)

        results.append({
            "tokens": visible_tokens,
            "pred_labels": pred_labels,
            "pred_confidences": pred_confidences,
            "entities": entities,
            "rendered": rendered,
            "was_truncated": len(tokens_with_offsets) > supervised_token_count,
            "original_token_count": len(tokens_with_offsets),
            "supervised_token_count": supervised_token_count,
        })

    return results

In [13]:
# ============================================================
# 13. Prueba rápida de inferencia
# ============================================================

manual_texts = [
    "The crab legs were amazing, but the fries were cold.",
    "I ordered the burger with bacon and the mac and cheese.",
    "The service was terrible, but the pasta was delicious.",
    "We loved the sushi, the salmon and the spicy tuna tacos.",
    "The atmosphere was nice, but the food was average.",
    "Their fried chicken and mashed potatoes were fantastic.",
    "I would come back just for the ice cream and the pumpkin pie.",
]

manual_results = predict_batch_texts(manual_texts)

manual_rows = []

for text, result in zip(manual_texts, manual_results):
    manual_rows.append({
        "text": text,
        "rendered": result["rendered"],
        "entities": result["entities"],
        "was_truncated": result["was_truncated"],
    })

manual_df = pd.DataFrame(manual_rows)

display(manual_df)

,text,rendered,entities,was_truncated
0,"The crab legs were amazing, but the fries were cold.","The[DISH:crab legs] were amazing , but the[DISH:fries] were cold .","[{'label': 'DISH', 'start_token': 1, 'end_token': 3, 'start_char': 4, 'end_char': 13, 'tokens': ['crab', 'legs'], 'token_confidences': [0.9981948733329773, 0.9993160963058472], 'text': 'crab legs', 'normalized_text': 'crab legs', 'confidence_mean': 0.9987554848194122, 'confidence_min': 0.9981948...",False
1,I ordered the burger with bacon and the mac and cheese.,I ordered the[DISH:burger with bacon] and the[DISH:mac and cheese] .,"[{'label': 'DISH', 'start_token': 3, 'end_token': 6, 'start_char': 14, 'end_char': 31, 'tokens': ['burger', 'with', 'bacon'], 'token_confidences': [0.9984074234962463, 0.9991644620895386, 0.9992210865020752], 'text': 'burger with bacon', 'normalized_text': 'burger with bacon', 'confidence_mean':...",False
2,"The service was terrible, but the pasta was delicious.","The service was terrible , but the[DISH:pasta] was delicious .","[{'label': 'DISH', 'start_token': 7, 'end_token': 8, 'start_char': 34, 'end_char': 39, 'tokens': ['pasta'], 'token_confidences': [0.8655649423599243], 'text': 'pasta', 'normalized_text': 'pasta', 'confidence_mean': 0.8655649423599243, 'confidence_min': 0.8655649423599243, 'confidence_max': 0.865...",False
3,"We loved the sushi, the salmon and the spicy tuna tacos.","We loved the[DISH:sushi] , the[DISH:salmon] and the spicy[DISH:tuna][DISH:tacos] .","[{'label': 'DISH', 'start_token': 3, 'end_token': 4, 'start_char': 13, 'end_char': 18, 'tokens': ['sushi'], 'token_confidences': [0.9932697415351868], 'text': 'sushi', 'normalized_text': 'sushi', 'confidence_mean': 0.9932697415351868, 'confidence_min': 0.9932697415351868, 'confidence_max': 0.993...",False
4,"The atmosphere was nice, but the food was average.","The atmosphere was nice , but the food was average .",[],False
5,Their fried chicken and mashed potatoes were fantastic.,Their[DISH:fried chicken] and[DISH:mashed potatoes] were fantastic .,"[{'label': 'DISH', 'start_token': 1, 'end_token': 3, 'start_char': 6, 'end_char': 19, 'tokens': ['fried', 'chicken'], 'token_confidences': [0.998976469039917, 0.9995476603507996], 'text': 'fried chicken', 'normalized_text': 'fried chicken', 'confidence_mean': 0.9992620646953583, 'confidence_min'...",False
6,I would come back just for the ice cream and the pumpkin pie.,I would come back just for the[DISH:ice cream] and the[DISH:pumpkin pie] .,"[{'label': 'DISH', 'start_token': 7, 'end_token': 9, 'start_char': 31, 'end_char': 40, 'tokens': ['ice', 'cream'], 'token_confidences': [0.9994500279426575, 0.9995484948158264], 'text': 'ice cream', 'normalized_text': 'ice cream', 'confidence_mean': 0.9994992613792419, 'confidence_min': 0.999450...",False


In [14]:
# ============================================================
# 14. Postprocesado de entidades detectadas
# ============================================================

def recompute_entity_confidence(entity: dict) -> dict:
    confidences = entity.get("token_confidences", [])

    if confidences:
        entity["confidence_mean"] = float(np.mean(confidences))
        entity["confidence_min"] = float(np.min(confidences))
        entity["confidence_max"] = float(np.max(confidences))
    else:
        entity["confidence_mean"] = None
        entity["confidence_min"] = None
        entity["confidence_max"] = None

    return entity


def merge_adjacent_dish_entities(entities: list[dict]) -> list[dict]:
    """
    Fusiona entidades DISH adyacentes sin tokens intermedios.

    Ejemplo:
    [DISH:tuna][DISH:tacos] -> [DISH:tuna tacos]

    No fusiona entidades separadas por coma, 'and', 'the', etc.
    """

    if not entities:
        return []

    entities = sorted(entities, key=lambda e: (e["start_token"], e["end_token"]))

    merged = []

    for entity in entities:
        entity = dict(entity)

        if not merged:
            merged.append(entity)
            continue

        previous = merged[-1]

        are_adjacent = previous["end_token"] == entity["start_token"]

        if are_adjacent:
            previous["end_token"] = entity["end_token"]
            previous["end_char"] = entity["end_char"]
            previous["tokens"] = previous.get("tokens", []) + entity.get("tokens", [])
            previous["token_confidences"] = (
                previous.get("token_confidences", [])
                + entity.get("token_confidences", [])
            )

            previous["text"] = " ".join(previous["tokens"])
            previous["normalized_text"] = normalize_mention_text(previous["text"])

            previous = recompute_entity_confidence(previous)
            merged[-1] = previous

        else:
            merged.append(entity)

    return merged


def normalize_mention_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = text.strip(" -_:;,.!?()[]{}\"'")

    return text


def postprocess_entities(entities: list[dict]) -> list[dict]:
    """
    Postprocesado ligero para la v1:
    - fusionar entidades DISH adyacentes;
    - normalizar texto;
    - recalcular confianza.
    """

    if not entities:
        return []

    processed = merge_adjacent_dish_entities(entities)

    final_entities = []

    for entity in processed:
        entity = dict(entity)

        entity["text"] = " ".join(entity.get("tokens", []))
        entity["normalized_text"] = normalize_mention_text(entity["text"])
        entity = recompute_entity_confidence(entity)

        if entity["normalized_text"]:
            final_entities.append(entity)

    return final_entities


# Reprobar sobre los textos manuales con postprocesado
manual_rows_postprocessed = []

for text, result in zip(manual_texts, manual_results):
    processed_entities = postprocess_entities(result["entities"])

    manual_rows_postprocessed.append({
        "text": text,
        "raw_entities": result["entities"],
        "postprocessed_entities": processed_entities,
    })

manual_postprocessed_df = pd.DataFrame(manual_rows_postprocessed)

display(manual_postprocessed_df)

,text,raw_entities,postprocessed_entities
0,"The crab legs were amazing, but the fries were cold.","[{'label': 'DISH', 'start_token': 1, 'end_token': 3, 'start_char': 4, 'end_char': 13, 'tokens': ['crab', 'legs'], 'token_confidences': [0.9981948733329773, 0.9993160963058472], 'text': 'crab legs', 'normalized_text': 'crab legs', 'confidence_mean': 0.9987554848194122, 'confidence_min': 0.9981948...","[{'label': 'DISH', 'start_token': 1, 'end_token': 3, 'start_char': 4, 'end_char': 13, 'tokens': ['crab', 'legs'], 'token_confidences': [0.9981948733329773, 0.9993160963058472], 'text': 'crab legs', 'normalized_text': 'crab legs', 'confidence_mean': 0.9987554848194122, 'confidence_min': 0.9981948..."
1,I ordered the burger with bacon and the mac and cheese.,"[{'label': 'DISH', 'start_token': 3, 'end_token': 6, 'start_char': 14, 'end_char': 31, 'tokens': ['burger', 'with', 'bacon'], 'token_confidences': [0.9984074234962463, 0.9991644620895386, 0.9992210865020752], 'text': 'burger with bacon', 'normalized_text': 'burger with bacon', 'confidence_mean':...","[{'label': 'DISH', 'start_token': 3, 'end_token': 6, 'start_char': 14, 'end_char': 31, 'tokens': ['burger', 'with', 'bacon'], 'token_confidences': [0.9984074234962463, 0.9991644620895386, 0.9992210865020752], 'text': 'burger with bacon', 'normalized_text': 'burger with bacon', 'confidence_mean':..."
2,"The service was terrible, but the pasta was delicious.","[{'label': 'DISH', 'start_token': 7, 'end_token': 8, 'start_char': 34, 'end_char': 39, 'tokens': ['pasta'], 'token_confidences': [0.8655649423599243], 'text': 'pasta', 'normalized_text': 'pasta', 'confidence_mean': 0.8655649423599243, 'confidence_min': 0.8655649423599243, 'confidence_max': 0.865...","[{'label': 'DISH', 'start_token': 7, 'end_token': 8, 'start_char': 34, 'end_char': 39, 'tokens': ['pasta'], 'token_confidences': [0.8655649423599243], 'text': 'pasta', 'normalized_text': 'pasta', 'confidence_mean': 0.8655649423599243, 'confidence_min': 0.8655649423599243, 'confidence_max': 0.865..."
3,"We loved the sushi, the salmon and the spicy tuna tacos.","[{'label': 'DISH', 'start_token': 3, 'end_token': 4, 'start_char': 13, 'end_char': 18, 'tokens': ['sushi'], 'token_confidences': [0.9932697415351868], 'text': 'sushi', 'normalized_text': 'sushi', 'confidence_mean': 0.9932697415351868, 'confidence_min': 0.9932697415351868, 'confidence_max': 0.993...","[{'label': 'DISH', 'start_token': 3, 'end_token': 4, 'start_char': 13, 'end_char': 18, 'tokens': ['sushi'], 'token_confidences': [0.9932697415351868], 'text': 'sushi', 'normalized_text': 'sushi', 'confidence_mean': 0.9932697415351868, 'confidence_min': 0.9932697415351868, 'confidence_max': 0.993..."
4,"The atmosphere was nice, but the food was average.",[],[]
5,Their fried chicken and mashed potatoes were fantastic.,"[{'label': 'DISH', 'start_token': 1, 'end_token': 3, 'start_char': 6, 'end_char': 19, 'tokens': ['fried', 'chicken'], 'token_confidences': [0.998976469039917, 0.9995476603507996], 'text': 'fried chicken', 'normalized_text': 'fried chicken', 'confidence_mean': 0.9992620646953583, 'confidence_min'...","[{'label': 'DISH', 'start_token': 1, 'end_token': 3, 'start_char': 6, 'end_char': 19, 'tokens': ['fried', 'chicken'], 'token_confidences': [0.998976469039917, 0.9995476603507996], 'text': 'fried chicken', 'normalized_text': 'fried chicken', 'confidence_mean': 0.9992620646953583, 'confidence_min'..."
6,I would come back just for the ice cream and the pumpkin pie.,"[{'label': 'DISH', 'start_token': 7, 'end_token': 9, 'start_char': 31, 'end_char': 40, 'tokens': ['ice', 'cream'], 'token_confidences': [0.9994500279426575, 0.9995484948158264], 'text': 'ice cream', 'normalized_text': 'ice cream', 'confidence_mean': 0.9994992613792419, 'confidence_min': 0.999450...","[{'label': 'DISH', 'start_token': 7, 'end_token': 9, 'start_char': 31, 'end_char': 40, 'tokens': ['ice', 'cream'], 'token_confidences': [0.9994500279426575, 0.9995484948158264], 'text': 'ice cream', 'normalized_text':

In [15]:
# ============================================================
# 15. Funciones auxiliares de guardado
# ============================================================

import hashlib

def make_json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, float) and np.isnan(value):
        return None

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, set):
        return sorted(list(value))

    return value


def save_jsonl(df_to_save: pd.DataFrame, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for record in df_to_save.to_dict(orient="records"):
            clean_record = {
                str(key): make_json_safe(value)
                for key, value in record.items()
            }

            f.write(json.dumps(clean_record, ensure_ascii=False) + "\n")

    print(f"Guardado: {path} ({len(df_to_save):,} registros)")


def build_mention_id(review_id: str, mention_index: int, start_char: int, end_char: int, mention_text: str) -> str:
    raw = f"{review_id}|{mention_index}|{start_char}|{end_char}|{mention_text}"
    digest = hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]
    return f"dish_mention_{digest}"

In [16]:
# ============================================================
# 16. Aplicar inferencia al dataset seleccionado
# ============================================================

all_review_prediction_rows = []
all_mention_rows = []

texts = df["text"].tolist()

print("Total reviews para inferencia:", len(df))
print("BATCH_SIZE:", BATCH_SIZE)
print("RUN_NAME:", RUN_NAME)

for start in tqdm(range(0, len(df), BATCH_SIZE), desc="Inferencia Dish NER"):
    end = min(start + BATCH_SIZE, len(df))

    batch_df = df.iloc[start:end].copy()
    batch_texts = batch_df["text"].tolist()

    batch_results = predict_batch_texts(batch_texts)

    for local_idx, (_, row) in enumerate(batch_df.iterrows()):
        result = batch_results[local_idx]

        raw_entities = result["entities"]
        postprocessed_entities = postprocess_entities(raw_entities)

        review_prediction_row = {
            "corpus_document_id": row["corpus_document_id"],
            "review_id": row["review_id"],
            "business_id": row["business_id"],
            "business_name": row.get("business_name", ""),
            "city": row.get("city", ""),
            "state": row.get("state", ""),
            "split": row.get("split", ""),
            "rating_value": make_json_safe(row.get("rating_value")),
            "sentiment_label": row.get("sentiment_label", ""),
            "language": row.get("language", ""),
            "text": row["text"],
            "text_length_chars": make_json_safe(row.get("text_length_chars")),
            "tokens": result["tokens"],
            "pred_labels": result["pred_labels"],
            "pred_confidences": result["pred_confidences"],
            "raw_entities": raw_entities,
            "postprocessed_entities": postprocessed_entities,
            "raw_entity_count": len(raw_entities),
            "postprocessed_entity_count": len(postprocessed_entities),
            "was_truncated": bool(result["was_truncated"]),
            "original_token_count": int(result["original_token_count"]),
            "supervised_token_count": int(result["supervised_token_count"]),
            "rendered": result["rendered"],
            "model_name": "dish_ner_transformer_full",
            "model_checkpoint": str(MODEL_PATH),
            "inference_run_name": RUN_NAME,
        }

        all_review_prediction_rows.append(review_prediction_row)

        for mention_index, entity in enumerate(postprocessed_entities):
            mention_text = entity.get("text", "")
            normalized_text = normalize_mention_text(mention_text)

            mention_row = {
                "mention_id": build_mention_id(
                    review_id=row["review_id"],
                    mention_index=mention_index,
                    start_char=int(entity["start_char"]),
                    end_char=int(entity["end_char"]),
                    mention_text=mention_text,
                ),
                "corpus_document_id": row["corpus_document_id"],
                "review_id": row["review_id"],
                "business_id": row["business_id"],
                "business_name": row.get("business_name", ""),
                "city": row.get("city", ""),
                "state": row.get("state", ""),
                "split": row.get("split", ""),
                "rating_value": make_json_safe(row.get("rating_value")),
                "sentiment_label": row.get("sentiment_label", ""),
                "language": row.get("language", ""),
                "source_system_code": row.get("source_system_code", "yelp_open_dataset"),
                "source_dataset": row.get("source_dataset", ""),
                "dish_mention_text": mention_text,
                "dish_mention_normalized": normalized_text,
                "start_char": int(entity["start_char"]),
                "end_char": int(entity["end_char"]),
                "start_token": int(entity["start_token"]),
                "end_token": int(entity["end_token"]),
                "token_count": int(entity["end_token"] - entity["start_token"]),
                "confidence_mean": make_json_safe(entity.get("confidence_mean")),
                "confidence_min": make_json_safe(entity.get("confidence_min")),
                "confidence_max": make_json_safe(entity.get("confidence_max")),
                "tokens": entity.get("tokens", []),
                "review_text": row["text"],
                "was_review_truncated": bool(result["was_truncated"]),
                "model_name": "dish_ner_transformer_full",
                "model_checkpoint": str(MODEL_PATH),
                "inference_run_name": RUN_NAME,
                "human_review_status": "not_reviewed",
                "normalization_status": "pending",
            }

            all_mention_rows.append(mention_row)

review_predictions_df = pd.DataFrame(all_review_prediction_rows)
dish_mentions_df = pd.DataFrame(all_mention_rows)

print("Reviews procesadas:", len(review_predictions_df))
print("Menciones detectadas:", len(dish_mentions_df))
print("Reviews con menciones:", review_predictions_df[review_predictions_df["postprocessed_entity_count"] > 0]["review_id"].nunique())

display(review_predictions_df.head(3))
display(dish_mentions_df.head(10))

Total reviews para inferencia: 79270
BATCH_SIZE: 32
RUN_NAME: full


Inferencia Dish NER:   0%|          | 0/2478 [00:00<?, ?it/s]

Reviews procesadas: 79270
Menciones detectadas: 95061
Reviews con menciones: 42471


,corpus_document_id,review_id,business_id,business_name,city,state,split,rating_value,sentiment_label,language,text,text_length_chars,tokens,pred_labels,pred_confidences,raw_entities,postprocessed_entities,raw_entity_count,postprocessed_entity_count,was_truncated,original_token_count,supervised_token_count,rendered,model_name,model_checkpoint,inference_run_name
0,yelp_2fbfd094613536a7b8c9231b,KU_O5udG6zpxOg-VcAEodg,XQfwVwDr-v0ZS3_CbbE5Xw,Turning Point of North Wales,North Wales,PA,train,3.0,neutral,en,"If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. The food is good, but it takes a very long time to come out. The...",511,"[If, you, decide, to, eat, here, ,, just, be, aware, it, is, going, to, take, about, 2, hours, from, beginning, to, end, ., We, have, tried, it, multiple, times, ,, because, I, want, to, like, it, !, I, have, been, to, it's, other, locations, in, NJ, and, never, had, a, bad, experience, ., The, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O...","[0.9999436140060425, 0.9999408721923828, 0.9999488592147827, 0.9999518394470215, 0.9999420642852783, 0.9999439716339111, 0.9998518228530884, 0.9999250173568726, 0.9999195337295532, 0.9999440908432007, 0.999951958656311, 0.9999473094940186, 0.9999444484710693, 0.9999518394470215, 0.99994218349456...",[],[],0,0,False,113,113,"If you decide to eat here , just be aware it is going to take about 2 hours from beginning to end . We have tried it multiple times , because I want to like it ! I have been to it's other locations in NJ and never had a bad experience . The food is good , but it takes a very long time to come ou...",dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full
1,yelp_94c5a64cecd4448d105e5c8a,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,Kettle Restaurant,Tucson,AZ,train,3.0,neutral,en,"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ...",339,"[Family, diner, ., Had, the, buffet, ., Eclectic, assortment, :, a, large, chicken, leg, ,, fried, jalapeño, ,, tamale, ,, two, rolled, grape, leaves, ,, fresh, melon, ., All, good, ., Lots, of, Mexican, choices, there, ., Also, has, a, menu, with, breakfast, served, all, day, long, ., Friendly,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-DISH, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[0.9999305009841919, 0.9999029636383057, 0.9999376535415649, 0.999943733215332, 0.9999325275421143, 0.9998430013656616, 0.9998946189880371, 0.9999459981918335, 0.9999356269836426, 0.9999376535415649, 0.999935507774353, 0.9979128241539001, 0.9976685643196106, 0.9842064380645752, 0.999942541122436...","[{'label': 'DISH', 'start_token': 18, 'end_token': 19, 'start_char': 88, 'end_char': 94, 'tokens': ['tamale'], 'token_confidences': [0.997416615486145], 'text': 'tamale', 'normalized_text': 'tamale', 'confidence_mean': 0.997416615486145, 'confidence_min': 0.997416615486145, 'confidence_max': 0.9...","[{'label': 'DISH', 'start_token': 18, 'end_token': 19, 'start_char': 88, 'end_char': 94, 'tokens': ['tamale'], 'token_confidences': [0.997416615486145], 'text': 'tamale', 'normalized_text': 'tamale', 'confidence_mean': 0.997416615486145, 'confidence_min': 0.997416615486145, 'c

,mention_id,corpus_document_id,review_id,business_id,business_name,city,state,split,rating_value,sentiment_label,language,source_system_code,source_dataset,dish_mention_text,dish_mention_normalized,start_char,end_char,start_token,end_token,token_count,confidence_mean,confidence_min,confidence_max,tokens,review_text,was_review_truncated,model_name,model_checkpoint,inference_run_name,human_review_status,normalization_status
0,dish_mention_c6a233f28ab0ec50a4bf,yelp_94c5a64cecd4448d105e5c8a,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,Kettle Restaurant,Tucson,AZ,train,3.0,neutral,en,yelp_open_dataset,yelp_open_dataset,tamale,tamale,88,94,18,19,1,0.997417,0.997417,0.997417,[tamale],"Family diner. Had the buffet. Eclectic assortment: a large chicken leg, fried jalapeño, tamale, two rolled grape leaves, fresh melon. All good. Lots of Mexican choices there. Also has a menu with breakfast served all day long. Friendly, attentive staff. Good place for a casual relaxed meal with ...",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
1,dish_mention_370e33f760028e4e87a3,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,lamb curry,lamb curry,54,64,12,14,2,0.996373,0.993416,0.999331,"[lamb, curry]","Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
2,dish_mention_2c51d464763786d9fef2,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,korma,korma,69,74,15,16,1,0.997466,0.997466,0.997466,[korma],"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
3,dish_mention_6cfa246893992204e58a,yelp_69e10d25d69774ab39af6571,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,Zaika,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,naan,naan,103,107,22,23,1,0.998627,0.998627,0.998627,[naan],"Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds of naan!!! Don't let the outside deter you (because we almost changed our minds)...go in and try something new! You'll be glad you did!",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
4,dish_mention_e468cc705c76b7c9e84f,yelp_95095495afc3a77ce68723a2,Sx8TMOWLNuJBWer-0pcmoA,e4Vwtrqf-wpJfwesgvdgxQ,Melt,New Orleans,LA,train,4.0,positive,en,yelp_open_dataset,yelp_open_dataset,sandwiches,sandwiches,185,195,38,39,1,0.999447,0.999447,0.999447,[sandwiches],"Cute interior and owner (?) gave us tour of upcoming patio/rooftop area which will be great on beautiful days like today. Cheese curds were very good and very filling. Really like that sandwiches come w salad, esp after eating too many curds! Had the onion, gruyere, tomato sandwich. Wasn't too m...",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
5,dish_mention_65f14f9a27141a174838,yelp_6ce3d2a264cfcdddc59ac240,_ZeMknuYdlQcUqng_Im3yg,LHSTtnW3YHCeUkRDGyJOyw,Fries Rebellion,Quakertown,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,wings,wings,18,2

In [17]:
# ============================================================
# 17. QA de menciones generadas
# ============================================================

if len(dish_mentions_df) == 0:
    raise ValueError("No se han generado menciones. Revisa el modelo o la función de inferencia.")

qa_summary = {
    "run_name": RUN_NAME,
    "fast_dev_run": bool(FAST_DEV_RUN),
    "total_reviews_processed": int(len(review_predictions_df)),
    "total_mentions_detected": int(len(dish_mentions_df)),
    "reviews_with_mentions": int(
        review_predictions_df[review_predictions_df["postprocessed_entity_count"] > 0]["review_id"].nunique()
    ),
    "reviews_without_mentions": int(
        review_predictions_df[review_predictions_df["postprocessed_entity_count"] == 0]["review_id"].nunique()
    ),
    "truncated_reviews": int(review_predictions_df["was_truncated"].sum()),
    "avg_mentions_per_review": float(len(dish_mentions_df) / len(review_predictions_df)),
    "avg_mentions_per_review_with_mentions": float(
        len(dish_mentions_df)
        / max(1, review_predictions_df[review_predictions_df["postprocessed_entity_count"] > 0]["review_id"].nunique())
    ),
    "confidence": {
        "mean": float(dish_mentions_df["confidence_mean"].mean()),
        "median": float(dish_mentions_df["confidence_mean"].median()),
        "min": float(dish_mentions_df["confidence_mean"].min()),
        "max": float(dish_mentions_df["confidence_mean"].max()),
    },
    "top_mentions": (
        dish_mentions_df["dish_mention_normalized"]
        .value_counts()
        .head(50)
        .to_dict()
    ),
    "split_counts_reviews": {
        str(k): int(v)
        for k, v in review_predictions_df["split"].value_counts().to_dict().items()
    },
    "split_counts_mentions": {
        str(k): int(v)
        for k, v in dish_mentions_df["split"].value_counts().to_dict().items()
    },
    "sentiment_counts_mentions": {
        str(k): int(v)
        for k, v in dish_mentions_df["sentiment_label"].value_counts().to_dict().items()
    },
}

print(json.dumps(qa_summary, indent=2, ensure_ascii=False)[:5000])

print("\nTop menciones:")
display(
    dish_mentions_df["dish_mention_normalized"]
    .value_counts()
    .head(50)
    .reset_index()
    .rename(columns={"index": "dish_mention_normalized", "dish_mention_normalized": "count"})
)

print("\nDistribución de confianza:")
display(dish_mentions_df["confidence_mean"].describe())

print("\nMenciones de baja confianza:")
display(
    dish_mentions_df
    .sort_values("confidence_mean", ascending=True)
    [
        [
            "dish_mention_text",
            "confidence_mean",
            "rating_value",
            "sentiment_label",
            "review_text"
        ]
    ]
    .head(20)
)

{
  "run_name": "full",
  "fast_dev_run": false,
  "total_reviews_processed": 79270,
  "total_mentions_detected": 95061,
  "reviews_with_mentions": 42471,
  "reviews_without_mentions": 36799,
  "truncated_reviews": 8279,
  "avg_mentions_per_review": 1.1992052478869686,
  "avg_mentions_per_review_with_mentions": 2.2382566928021475,
  "confidence": {
    "mean": 0.975224027584393,
    "median": 0.9990728795528412,
    "min": 0.35621654987335205,
    "max": 0.9997578263282776
  },
  "top_mentions": {
    "pizza": 8682,
    "burger": 5756,
    "fries": 4848,
    "sushi": 4461,
    "shrimp": 4398,
    "tacos": 2912,
    "steak": 2789,
    "ice cream": 2371,
    "burgers": 2334,
    "wings": 2213,
    "sandwiches": 2034,
    "oysters": 1691,
    "pancakes": 1536,
    "crab": 1530,
    "taco": 1475,
    "donuts": 1419,
    "pho": 1219,
    "salmon": 1134,
    "ribs": 1093,
    "burrito": 993,
    "fried chicken": 933,
    "tuna": 889,
    "french toast": 862,
    "donut": 756,
    "brisket": 

,count,count
0,pizza,8682
1,burger,5756
2,fries,4848
3,sushi,4461
4,shrimp,4398
5,tacos,2912
6,steak,2789
7,ice cream,2371
8,burgers,2334
9,wings,2213



Distribución de confianza:


count    95061.000000
mean         0.975224
std          0.079815
min          0.356217
25%          0.997282
50%          0.999073
75%          0.999427
max          0.999758
Name: confidence_mean, dtype: float64


Menciones de baja confianza:


,dish_mention_text,confidence_mean,rating_value,sentiment_label,review_text
19337,tempeh,0.356217,4.0,positive,"Great little Indonesian restaurant near Cacias bakery on corner of Ritner & 16th. There is a sign inside warning not to park in front of Cacias. Mostly Indonesian clientele. Food was really good and came out fast. Menu is divided into noodle(w/ broth or stir-fried), rice dishes, a few starters a..."
71605,loin,0.366022,3.0,neutral,My wife and I decided to give Workshop Eatery a try after a friend recommended it. We arrived 5:15pm on a Thursday and were promptly seated. The server asked for our drink order and delivered it 10 minutes later at which time she took our order. We started with the Arancini which was very good m...
2599,bun,0.397663,4.0,positive,"Spot has been on my burger list to try for sometime. Spot burger consistently shows up on numerous best of Philly burger lists. When they opened up shop in Brewery town, I quickly bookmarked Spot and waited for a burger craving to hit me to give it a try. Spot is on West Girard Ave and I was abl..."
32966,noodle,0.398607,5.0,positive,"I've been going here since I was a kid. The same waitress that served me twenty years ago is still there. Fast, courteous, even too polite at times. Went today for lunch to get my favorite dish- Taiwanese beef noodle soup. This soup is the best; it certainly holds up against the stuff I've had i..."
8814,cupcake,0.404182,4.0,positive,"We were shopping in Indianapolis on recent trip and craving something sweet. We saw The Flying Cupcake and stopped by on the way back from the mall. It was around 6 or 7 on a Saturday afternoon and there was not many cupcakes left choose from. I think they had wedding cake cupcake, lemon, bluebe..."
64529,salad,0.414822,5.0,positive,My mom and I had a delicious meal here! We are from New Jersey and wanted some good food after a day of travel. We read great reviews and were not disappointed. We started with the Carpaccio which was amazing!! I got the B&M burger for my meal and it had a delicious combination with the aged che...
39284,pinwheel,0.434031,4.0,positive,"I'm originally from Brooklyn, and am an Italian American. I really like Ceriello's. I discovered it first at their location in Grand Central, and was happy to discover their location in Medford, way closer to where I live. Great meats, including Veal, but our favorite is the Brooklyn/Long Island..."
52084,sandwich,0.438024,5.0,positive,"The food and service was excellent. We had wings, Portobello pesto sandwich with sweet potato fries, and an asparagus pasta dish. These were the best wings I ever had, right off the grill. The sandwich was delicious and the pasta dish was creamy with great blend of flavors. Next time I want to t..."
45154,tomatos,0.447881,4.0,positive,Hooray for YELP. How did I get by before Yelp? I was up in Santa Barbara the other day and we noticed this Cafe as we drove around. The next day when we wanted to go out for breakfast I went to YELP to read the reviews. Its rating was good enough for us to go try it and we are so happy we did. T...
57500,Cake,0.455745,4.0,positive,"Really enjoyed this New Orleans gem on Sunday, it was recommended to us by our amazing cocktail tour guide from Drink & Learn. She said this was a favorite place of hers for live music. We walked from our hotel on Canal Street to Frenchman Street and enjoyed the leisurely stroll on some of the n..."


In [18]:
# ============================================================
# 18. Crear muestras de revisión
# ============================================================

# Muestra aleatoria general
sample_mentions_random_df = dish_mentions_df.sample(
    n=min(200, len(dish_mentions_df)),
    random_state=RANDOM_STATE
).copy()

# Muestra de baja confianza
sample_mentions_low_conf_df = (
    dish_mentions_df
    .sort_values("confidence_mean", ascending=True)
    .head(min(200, len(dish_mentions_df)))
    .copy()
)

# Muestra de menciones frecuentes
top_mention_names = (
    dish_mentions_df["dish_mention_normalized"]
    .value_counts()
    .head(30)
    .index
    .tolist()
)

sample_mentions_top_df = (
    dish_mentions_df[dish_mentions_df["dish_mention_normalized"].isin(top_mention_names)]
    .groupby("dish_mention_normalized", group_keys=False)
    .head(5)
    .copy()
)

print("Sample random:", len(sample_mentions_random_df))
print("Sample low confidence:", len(sample_mentions_low_conf_df))
print("Sample top mentions:", len(sample_mentions_top_df))

display(sample_mentions_random_df.head(5))
display(sample_mentions_low_conf_df.head(5))
display(sample_mentions_top_df.head(5))

Sample random: 200
Sample low confidence: 200
Sample top mentions: 150


,mention_id,corpus_document_id,review_id,business_id,business_name,city,state,split,rating_value,sentiment_label,language,source_system_code,source_dataset,dish_mention_text,dish_mention_normalized,start_char,end_char,start_token,end_token,token_count,confidence_mean,confidence_min,confidence_max,tokens,review_text,was_review_truncated,model_name,model_checkpoint,inference_run_name,human_review_status,normalization_status
12400,dish_mention_625640757e831e245e9f,yelp_81ad48c38705e9b15ccf6f88,zkhcJ9VC11ypscHjNmSJDA,1IjHo5kaSdFxJkJotU4O7A,Sociale Italian Tapas & Pizza Bar,Tampa,FL,train,2.0,negative,en,yelp_open_dataset,yelp_open_dataset,eggplant,eggplant,311,319,64,65,1,0.999137,0.999137,0.999137,[eggplant],"I thought the food was terribly greasy. Even the plate of broccoli and cauliflower left a coating of oil on the bowl. The sausage and potatoes were soaked in oil--the potatoes were browned nicely, but the sausage had no color on them and the onions weren't sautéed well. The the only thing I coul...",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
49482,dish_mention_20df9bc5498931843cfa,yelp_982aaba4f528706b8449d875,ELadbDPCwlQnZb2dQsn5OA,7apWV3_bxbRcC2MemII9dQ,Pan D'olive,Saint Louis,MO,validation,4.0,positive,en,yelp_open_dataset,yelp_open_dataset,salmon,salmon,319,325,68,69,1,0.999335,0.999335,0.999335,[salmon],"Great customer service! Food was very good, for the most part. Olive bread was good, but the olive oil the provided for dipping didn't meet expectations. It seemed a bit old...stick to the butter. Salad was overdressed but good portions and came split, a good service point. The Arancini were ver...",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
51161,dish_mention_ad27eabae9db8ee198d6,yelp_bd889c31d6d5abba76686c24,rQ1fh1AsepSQexrLO2H2cA,35GCFtezJ3upyqGf-zJ8MA,Gateway To India,St. Petersburg,FL,train,3.0,neutral,en,yelp_open_dataset,yelp_open_dataset,korma,korma,443,448,91,92,1,0.993231,0.993231,0.993231,[korma],I had such high hopes I could find an Indian restaurant I could frequent in st Pete. It took a long search to finally find 1 in tampa that I now frequently order takeout from. I found this place thru a Yelp article ranking the top Indian restaurants in Pinnellas. Real disappointed by this place....,True,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
9051,dish_mention_69800bf6bc2063a30999,yelp_78e944f28d162560f0f11bdd,FS4ewsVRMHvTvKBVRkX8ew,ADgeB1sfOGbzCR3LIi-4lg,Cafe Reconcile,New Orleans,LA,train,4.0,positive,en,yelp_open_dataset,yelp_open_dataset,mac and cheese,mac and cheese,426,440,77,80,3,0.999025,0.998709,0.999233,"[mac, and, cheese]","I didn't really read the reviews of this place before coming in, so I didn't even notice that this was a foundation for at-risk youth to learn culinary skills and hospitality. The place was clean and quaint and decorated well with such great service and courteousness from everyone at the establi...",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
35113,dish_mention_3d4c951ec62b15eac590,yelp_84c4b62ca28457c98f3cc9fa,IjLgnUtKBrZbOAt3JF7_cQ,lTVHJAvtFQbQb6cPstEXyA,China Bowl & State Street Cafe,Santa Barbara,CA,train,1.0,negative,en,yelp_open_dataset,yelp_open_dataset,wonton soup,wonton soup,98,109,20,22,2,0.997989,0.996719,0.999260,"[wonton, soup]","I walked here for lunch the other day after months of walking by and being curious. I ordered the wonton soup and some egg rolls. I love wonton soup, I have grown up loving it. I can not stress enough or be clear enough on how horrible this soup was. It was literally hot water with a ton of beef...",False,dish_ner_transformer_full,/kaggle/input

,mention_id,corpus_document_id,review_id,business_id,business_name,city,state,split,rating_value,sentiment_label,language,source_system_code,source_dataset,dish_mention_text,dish_mention_normalized,start_char,end_char,start_token,end_token,token_count,confidence_mean,confidence_min,confidence_max,tokens,review_text,was_review_truncated,model_name,model_checkpoint,inference_run_name,human_review_status,normalization_status
19337,dish_mention_54624608d71a1e8d2832,yelp_39a61463c6f167fdd244d064,_Pj3AEJ5DU2ScrnJwmwkFQ,6CsLAjWXFlLpErBZEmGQ5g,Sky Cafe,Philadelphia,PA,train,4.0,positive,en,yelp_open_dataset,yelp_open_dataset,tempeh,tempeh,479,485,100,101,1,0.356217,0.356217,0.356217,[tempeh],"Great little Indonesian restaurant near Cacias bakery on corner of Ritner & 16th. There is a sign inside warning not to park in front of Cacias. Mostly Indonesian clientele. Food was really good and came out fast. Menu is divided into noodle(w/ broth or stir-fried), rice dishes, a few starters a...",True,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
71605,dish_mention_8f89d5b7fa9c0a6ef9b0,yelp_7619577b390c28556a51de64,C30mh2VJ5VmCGdtqvYILVw,1LVMlrAQeqLgiwFu06kFfw,The Workshop Eatery,Edmonton,AB,validation,3.0,neutral,en,yelp_open_dataset,yelp_open_dataset,loin,loin,718,722,144,145,1,0.366022,0.366022,0.366022,[loin],My wife and I decided to give Workshop Eatery a try after a friend recommended it. We arrived 5:15pm on a Thursday and were promptly seated. The server asked for our drink order and delivered it 10 minutes later at which time she took our order. We started with the Arancini which was very good m...,False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
2599,dish_mention_9827050706b82291ce95,yelp_6d425329644c5061f553c795,wKVScyIiwc3cFwyNqLedAw,Gw7UW0E2BguzL9suQnwDeg,SPOT Gourmet Burgers,Philadelphia,PA,train,4.0,positive,en,yelp_open_dataset,yelp_open_dataset,bun,bun,967,970,202,203,1,0.397663,0.397663,0.397663,[bun],"Spot has been on my burger list to try for sometime. Spot burger consistently shows up on numerous best of Philly burger lists. When they opened up shop in Brewery town, I quickly bookmarked Spot and waited for a burger craving to hit me to give it a try. Spot is on West Girard Ave and I was abl...",True,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
32966,dish_mention_f83f8e98e3b0e17e5d99,yelp_2f79c62ace9b56eac5db0609,T-qChKcyxWjU5qZEEd4kxA,mCo2uVTTGYrEhRrkQW-CMw,Empress Garden,Philadelphia,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,noodle,noodle,210,216,45,46,1,0.398607,0.398607,0.398607,[noodle],"I've been going here since I was a kid. The same waitress that served me twenty years ago is still there. Fast, courteous, even too polite at times. Went today for lunch to get my favorite dish- Taiwanese beef noodle soup. This soup is the best; it certainly holds up against the stuff I've had i...",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
8814,dish_mention_fd5bea25ff9f98d483b8,yelp_b5fc4ee8f1590138c53aef9d,TeARi2ytr48ynz36u-1ilA,MnxMt3CErrx4dXll4ADqLg,The Flying Cupcake,Indianapolis,IN,train,4.0,positive,en,yelp_open_dataset,yelp_open_dataset,cupcake,cupcake,395,402,79,80,1,0.404182,0.404182,0.404182,[cupcake],"We were shopping in Indianapolis on recent trip and craving something sweet. We saw The Flying Cupcake and stopped by on the way back from the mall. It was around 6 or 7 on a Saturday afternoon and there was not many cupcakes left choose from. I think they had wedding cake cupcake, lemon, bluebe...",True,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full

,mention_id,corpus_document_id,review_id,business_id,business_name,city,state,split,rating_value,sentiment_label,language,source_system_code,source_dataset,dish_mention_text,dish_mention_normalized,start_char,end_char,start_token,end_token,token_count,confidence_mean,confidence_min,confidence_max,tokens,review_text,was_review_truncated,model_name,model_checkpoint,inference_run_name,human_review_status,normalization_status
4,dish_mention_e468cc705c76b7c9e84f,yelp_95095495afc3a77ce68723a2,Sx8TMOWLNuJBWer-0pcmoA,e4Vwtrqf-wpJfwesgvdgxQ,Melt,New Orleans,LA,train,4.0,positive,en,yelp_open_dataset,yelp_open_dataset,sandwiches,sandwiches,185,195,38,39,1,0.999447,0.999447,0.999447,[sandwiches],"Cute interior and owner (?) gave us tour of upcoming patio/rooftop area which will be great on beautiful days like today. Cheese curds were very good and very filling. Really like that sandwiches come w salad, esp after eating too many curds! Had the onion, gruyere, tomato sandwich. Wasn't too m...",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
5,dish_mention_65f14f9a27141a174838,yelp_6ce3d2a264cfcdddc59ac240,_ZeMknuYdlQcUqng_Im3yg,LHSTtnW3YHCeUkRDGyJOyw,Fries Rebellion,Quakertown,PA,train,5.0,positive,en,yelp_open_dataset,yelp_open_dataset,wings,wings,18,23,2,3,1,0.999090,0.999090,0.999090,[wings],"Amazingly amazing wings and homemade bleu cheese. Had the ribeye: tender, perfectly prepared, delicious. Nice selection of craft beers. Would DEFINITELY recommend checking out this hidden gem.",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
6,dish_mention_e03051cdd8bad08429a1,yelp_1e7a38ac7da3467ee929e559,pUycOfUwM8vqX7KjRRhUEA,gebiRewfieSdtt17PTW6Zg,Hibachi Steak House & Sushi Bar,Santa Barbara,CA,validation,3.0,neutral,en,yelp_open_dataset,yelp_open_dataset,sushi,sushi,69,74,14,15,1,0.999515,0.999515,0.999515,[sushi],Had a party of 6 here for hibachi. Our waitress brought our separate sushi orders on one plate so we couldn't really tell who's was who's and forgot several items on an order. I understand making mistakes but the restaraunt was really quiet so we were kind of surprised. Usually hibachi is a fun ...,False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
16,dish_mention_cfbd5e036c2d615fd59e,yelp_cd4740512bfa5a05e7020aa3,ZVvhc3Go7v5I8XTiVoWmqQ,ut6fi2W2YaipNOqvi7e0jw,Upland Carmel Tap House,Carmel,IN,train,3.0,neutral,en,yelp_open_dataset,yelp_open_dataset,burgers,burgers,209,216,42,43,1,0.999447,0.999447,0.999447,[burgers],"Upland is a brewery based out of Bloomington, Indiana that has become popular enough to open up a couple additional locations in central Indiana. All of their beers are very good, and I am also a fan of their burgers and tenderloins. Therefore, I was excited to try their pizza, but I don't think...",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending
17,dish_mention_eb7970aef6f3cab9b308,yelp_cd4740512bfa5a05e7020aa3,ZVvhc3Go7v5I8XTiVoWmqQ,ut6fi2W2YaipNOqvi7e0jw,Upland Carmel Tap House,Carmel,IN,train,3.0,neutral,en,yelp_open_dataset,yelp_open_dataset,pizza,pizza,272,277,54,55,1,0.999582,0.999582,0.999582,[pizza],"Upland is a brewery based out of Bloomington, Indiana that has become popular enough to open up a couple additional locations in central Indiana. All of their beers are very good, and I am also a fan of their burgers and tenderloins. Therefore, I was excited to try their pizza, but I don't think...",False,dish_ner_transformer_full,/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_transformer_full,full,not_reviewed,pending


In [19]:
# ============================================================
# 19. Guardar artefactos de inferencia
# ============================================================

mentions_path = MENTIONS_DIR / f"dish_mentions_model_v1_{RUN_NAME}.jsonl"
review_predictions_path = PREDICTIONS_DIR / f"dish_ner_review_predictions_model_v1_{RUN_NAME}.jsonl"
summary_path = ARTIFACTS_DIR / f"dish_ner_inference_summary_{RUN_NAME}.json"

sample_random_path = SAMPLES_DIR / f"dish_mentions_sample_random_{RUN_NAME}.jsonl"
sample_low_conf_path = SAMPLES_DIR / f"dish_mentions_sample_low_confidence_{RUN_NAME}.jsonl"
sample_top_path = SAMPLES_DIR / f"dish_mentions_sample_top_mentions_{RUN_NAME}.jsonl"

save_jsonl(dish_mentions_df, mentions_path)
save_jsonl(review_predictions_df, review_predictions_path)
save_jsonl(sample_mentions_random_df, sample_random_path)
save_jsonl(sample_mentions_low_conf_df, sample_low_conf_path)
save_jsonl(sample_mentions_top_df, sample_top_path)

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(qa_summary, f, indent=2, ensure_ascii=False)

print("Artefactos guardados:")
print("mentions_path:", mentions_path)
print("review_predictions_path:", review_predictions_path)
print("summary_path:", summary_path)
print("sample_random_path:", sample_random_path)
print("sample_low_conf_path:", sample_low_conf_path)
print("sample_top_path:", sample_top_path)

Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_inference/mentions/dish_mentions_model_v1_full.jsonl (95,061 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_inference/predictions/dish_ner_review_predictions_model_v1_full.jsonl (79,270 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_inference/samples/dish_mentions_sample_random_full.jsonl (200 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_inference/samples/dish_mentions_sample_low_confidence_full.jsonl (200 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_inference/samples/dish_mentions_sample_top_mentions_full.jsonl (150 registros)
Artefactos guardados:
mentions_path: /kaggle/working/hidden_gems_ai/outputs/dish_ner_inference/mentions/dish_mentions_model_v1_full.jsonl
review_predictions_path: /kaggle/working/hidden_gems_ai/outputs/dish_ner_inference/predictions/dish_ner_review_predictions_model_v1_full.jsonl
summary_path: /kaggle/working/hidde

In [20]:
# ============================================================
# 20. Comprimir outputs importantes
# ============================================================

zip_base = Path("/kaggle/working") / f"dish_ner_inference_outputs_{RUN_NAME}"

if zip_base.with_suffix(".zip").exists():
    zip_base.with_suffix(".zip").unlink()

shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(OUTPUT_DIR)
)

print("ZIP generado:")
print(str(zip_base) + ".zip")

ZIP generado:
/kaggle/working/dish_ner_inference_outputs_full.zip


## Cierre del Notebook 07

En este notebook se ha aplicado el modelo `Dish NER v1` sobre el corpus Yelp y se ha generado una tabla estructurada de menciones de platos.

Artefactos principales:

- `dish_mentions_model_v1_<run>.jsonl`
- `dish_ner_review_predictions_model_v1_<run>.jsonl`
- `dish_ner_inference_summary_<run>.json`
- muestras de revisión manual

El archivo principal para el siguiente módulo es:

`dish_mentions_model_v1_full.jsonl`

Este archivo será la entrada del módulo de normalización de platos:

`08_dish_normalization_and_catalog_builder.ipynb`